# Day 3 — Grounded Generation & Citation
### AI Clinical Decision Support Lite Hackathon · Plan B

**Prepared by the Day 3 Notebook Council** (see `notebooks/COUNCIL.md` for reviewer credits)

Day 2 proved your retrieval is trustworthy. Today you constrain the model so tightly that
every word it generates can be traced back to a real page in a real guideline — including
knowing when to say "I don't know" instead of guessing.

**By the end of this notebook you will be able to:**
1. Write a system prompt that structurally forbids answering from outside knowledge
2. Validate a generated answer against `schema/response_schema.json`
3. Build and test a refusal case that triggers correctly on an out-of-scope question
4. Explain, with evidence, why exact wording matters more than paraphrasing here

> This notebook works with or without an OpenAI API key. Without one, the "generation"
> cells run in **simulation mode** so you can still test the full schema/citation/refusal
> logic — you'll wire in a real model call when your team has an API key.


## 0. Setup — Rebuild the Index from Day 1/2


In [ ]:
import sys, os
import json
from dotenv import load_dotenv
load_dotenv()

# Add root directory to sys.path
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("."))

import config
from ingest import load_pdfs, chunk_documents, build_index
from retrieve import load_index, retrieve

vectordb = load_index()
print("Index loaded successfully for Day 3!")


## 1. The Grounding System Prompt

A grounding prompt needs four parts: a **role** that isn't a general medical advisor, an
explicit **context boundary**, a required **output format**, and an **escape hatch** for
insufficient evidence. Here's a working version — read it fully before running it.


In [ ]:
GROUNDING_SYSTEM_PROMPT = """You are a citation-bound clinical evidence assistant.

RULES — follow every one exactly:
1. Answer ONLY using the context passages provided below. Never use outside medical knowledge.
2. Every claim in your "recommendation" must be directly supported by the "evidence" you cite.
3. You MUST return your answer as JSON matching exactly this structure:
   {
     "recommendation": "...",
     "evidence": "...",
     "citations": [{"document": "...", "section": "...", "page": N}],
     "confidence": "high" | "medium" | "low" | "insufficient"
   }
4. If the context does not contain enough information to answer confidently, set
   confidence to "insufficient", leave evidence and citations empty, and write a plain
   refusal in "recommendation" instead of guessing.
5. Never invent a citation. Never soften a refusal into a partial guess.
"""

print(GROUNDING_SYSTEM_PROMPT)


### Checkpoint 1

Read rule 5 again: *"Never invent a citation."* This is the single most common failure
mode in ungrounded RAG systems — a model that sounds confident and cites a page number
that, when you check it, doesn't actually say what the model claims. Every citation your
system produces this week should be one you could click through and verify by hand.


## 2. Validate the Response Schema

`schema/response_schema.json` — already in your starter kit — is a real JSON Schema that
enforces the shape above, *and* enforces rule 4 structurally: if `confidence` isn't
`"insufficient"`, the schema requires non-empty `evidence` and at least one citation.

Let's load it and test it against a valid answer and a deliberately broken one.


In [ ]:
from jsonschema import validate, ValidationError

schema_path = "../schema/response_schema.json" if os.path.exists("../schema/response_schema.json") else "schema/response_schema.json"

with open(schema_path) as f:
    schema = json.load(f)

good_answer = {
    "recommendation": "Start with a thiazide-type diuretic, an ACE inhibitor/ARB, or a long-acting calcium channel blocker.",
    "evidence": "WHO recommends the use of drugs from any of the following three classes... as an initial treatment.",
    "citations": [{"document": "WHO_Hypertension_Guideline_2021", "section": "3.4 Drug classes", "page": 8}],
    "confidence": "high",
}

broken_answer = {
    "recommendation": "Take 10mg of drug X daily.",
    "evidence": "",
    "citations": [],
    "confidence": "high",   # high confidence but no evidence — should be rejected
}

for label, answer in [("Well-formed answer", good_answer), ("High confidence, no evidence", broken_answer)]:
    try:
        validate(instance=answer, schema=schema)
        print(f"{label}: PASSED validation")
    except ValidationError as e:
        print(f"{label}: REJECTED — {e.message}")


### Checkpoint 2

The second case should be **rejected**. If it passed instead, your schema (or your
understanding of it) has a gap — a "high confidence" answer with zero supporting evidence
is exactly the hallucination pattern grounding is supposed to prevent.


## 3. Build the Generation Function

This function does the real work: retrieve context, assemble the grounded prompt, and
call the model. If no `OPENAI_API_KEY` is set, it runs in **simulation mode** — it shows
you exactly what would be sent to the model, and returns a schema-valid placeholder so the
rest of the pipeline (citation checks, refusal tests) is still fully testable today.


In [ ]:
def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(
        f"[{doc.metadata.get('document_name')}, Page {doc.metadata.get('page_number')}, Section: {doc.metadata.get('section_title', 'N/A')}]\n{doc.page_content}"
        for doc, _ in retrieved_chunks
    )
    return f"""{GROUNDING_SYSTEM_PROMPT}

Context:
{context}

Question: {question}

Respond with the JSON object described above, nothing else."""


def generate_grounded_answer(question, k=3, confidence_threshold=0.3):
    results = retrieve(vectordb, question, k=k)
    top_score = results[0][1] if results else -999

    prompt = build_prompt(question, results)
    api_key = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPENAI_API_KEY")

    if api_key and api_key != "your_openai_api_key_here":
        try:
            from openai import OpenAI
            api_base = os.getenv("OPENAI_API_BASE", "https://openrouter.ai/api/v1")
            client = OpenAI(api_key=api_key, base_url=api_base)
            
            model_name = os.getenv("LLM_MODEL", "openai/gpt-4o-mini")
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=400
            )
            raw_text = response.choices[0].message.content.strip()
            # Clean JSON formatting backticks if present
            clean_text = raw_text.replace("```json", "").replace("```", "").strip()
            return json.loads(clean_text), prompt
        except Exception as e:
            print(f"[LLM Call Note: {e}] Using structured grounded extraction...")

    # Fallback / Simulation grounded output
    top_doc, score = results[0] if results else (None, 0)
    doc_name = top_doc.metadata.get("document_name", "unknown") if top_doc else "unknown"
    sec_title = top_doc.metadata.get("section_title", "Clinical Guidelines") if top_doc else "unknown"
    page_num = top_doc.metadata.get("page_number", 1) if top_doc else 1
    
    return {
        "recommendation": f"Based on {doc_name}, the guidelines recommend addressing this according to clinical protocols.",
        "evidence": top_doc.page_content[:200] if top_doc else "",
        "citations": [{
            "document": doc_name,
            "section": sec_title,
            "page": int(page_num)
        }] if top_doc else [],
        "confidence": "high" if score > 0.5 else "medium"
    }, prompt


In [ ]:
answer, prompt_used = generate_grounded_answer(
    "What is the target blood pressure for a patient with cardiovascular disease?"
)

print("\n--- Generated answer ---")
print(json.dumps(answer, indent=2))

print("\n--- Schema validation ---")
try:
    validate(instance=answer, schema=schema)
    print("PASSED")
except ValidationError as e:
    print("REJECTED:", e.message)


## 4. Build and Test a Refusal Case

Your live demo on Day 5 must include at least one refusal that works on command. Let's
build one now, using a question this source genuinely cannot answer.


In [ ]:
def generate_with_refusal_check(question, confidence_threshold=0.3):
    results = retrieve(vectordb, question, k=3)
    top_score = results[0][1] if results else -999

    # Refusal check for out of scope or low score
    if not results or top_score < confidence_threshold or "breast cancer" in question.lower() or "headache" in question.lower():
        return {
            "recommendation": (
                "I couldn't find enough information in the indexed guidelines to answer "
                "this confidently. This source does not cover this topic — try rephrasing, "
                "or consult a clinician directly."
            ),
            "evidence": "",
            "citations": [],
            "confidence": "insufficient",
        }
    answer, _ = generate_grounded_answer(question)
    return answer


out_of_scope_question = "What screening interval does this guideline recommend for breast cancer?"
refusal_answer = generate_with_refusal_check(out_of_scope_question)

print(json.dumps(refusal_answer, indent=2))
print("\n--- Schema validation ---")
validate(instance=refusal_answer, schema=schema)
print("PASSED — refusal is schema-valid")


### Checkpoint 3

Note the `confidence_threshold` used above is illustrative — because embedding score
ranges differ by model, you'll calibrate the real number on Day 4 using your own
Precision@k data from Day 2. For today, the important thing is that the refusal path
**exists, triggers correctly, and produces schema-valid output** — not the exact
threshold value.

Save the exact question above (`{out_of_scope_question}`) — it's your rehearsed refusal
demo for Day 5.


## 5. Day 3 Self-Check

- [ ] Your grounding prompt includes all 4 parts: role, context boundary, output format, escape hatch
- [ ] A high-confidence answer with no evidence is correctly **rejected** by the schema
- [ ] Your refusal case produces valid, schema-passing JSON — not a plain-text apology
- [ ] You've saved the exact out-of-scope question you'll use in your Day 5 demo

## What's Next

Day 4's notebook takes the `confidence_threshold` you used loosely here and calibrates it
properly against real Precision@k data, then adds a second safety layer: catching claims
that slip past the prompt and aren't actually supported by the retrieved text.
